# 04 — Evidence-Based Modelling Scope Selection

**Purpose.** Select a defensible initial agency, complaint-type population,
and complete date range for predicting whether a newly created NYC 311
complaint will miss its expected resolution target. The decision uses official
API-backed aggregates and the target contract established by Notebook 03.
Notebook 05 owns outcome-maturity analysis and the final date-range decision;
this notebook preserves the agency and complaint-subgroup feasibility stages.

## 2. Decision questions

Which agencies have enough deadline and closure information to construct the
approved target? Within the strongest agency, which complaint types and date
ranges have sufficient volume, continuity, class balance, and data quality?
What evidence supports approval—or blocks it?

## 3. Scope boundaries and non-goals

This notebook compares and documents scope. It does **not** train a model,
create train/validation/test splits, perform full EDA, invent a proxy deadline,
or redefine target eligibility. Thresholds below are configurable decision
rules for Month 1, not universal scientific constants.

## 4. Imports

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
import os
from pathlib import Path
import sys
from urllib.parse import urlencode

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd

## 5. Configuration

In [2]:
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "urban_ops").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from urban_ops.analysis.scope_selection import (
    SCORE_WEIGHTS,
    apply_agency_discovery_rules,
    apply_complaint_type_decision_rules,
    build_candidate_rejection_reasons,
    build_rejection_reasons,
    determine_scope_decision_status,
    evaluate_candidate_critical_gates,
    rebuild_scoped_sensitivity_candidate,
    score_scope_candidates,
    summarize_monthly_stability,
    validate_selected_scope_consistency,
    write_scope_decision,
)
from urban_ops.analysis.final_scope_selection import (
    reconcile_final_scope,
    validate_selected_scope_outputs,
)
from urban_ops.data.api_client import fetch_json_url
from urban_ops.data.nyc_311_config import (
    API_ENDPOINT,
    API_MAX_ATTEMPTS,
    API_TIMEOUT_SECONDS,
)
from urban_ops.utils.paths import ensure_report_directories, notebook_report_paths

NOTEBOOK_SLUG = "04_scope_selection"
REPORT_ROOT, TABLE_DIR, FIGURE_DIR = notebook_report_paths(NOTEBOOK_SLUG)
_, TEMPORAL_TABLE_DIR, _ = notebook_report_paths("05_temporal_stability")
SCOPE_DECISION_PATH = PROJECT_ROOT / "docs" / "scope_decision.md"

ANALYSIS_START_DATE = "2020-01-01"
ANALYSIS_END_DATE = None
MIN_AGENCY_ELIGIBLE_RECORDS = 10_000
MIN_AGENCY_DUE_DATE_COVERAGE = 0.70
MIN_AGENCY_CLOSED_DATE_COVERAGE = 0.80
MIN_AGENCY_ACTIVE_MONTHS = 12
MIN_TARGET_RATE = 0.05
MAX_TARGET_RATE = 0.95
MAX_INVALID_TIMESTAMP_RATE = 0.05
MIN_COMPLAINT_TYPE_ELIGIBLE_RECORDS = 1_000
MIN_COMPLAINT_TYPE_ACTIVE_MONTHS = 12
MIN_COMPLAINT_TYPE_MEDIAN_MONTHLY_VOLUME = 25
RECENT_ACTIVITY_MONTHS = 12
OPERATIONAL_RELEVANCE_SCORE = 0.50
GROUP_RESULT_LIMIT = 50_000

SCOPE_SELECTION_COLUMNS = [
    "unique_key", "created_date", "closed_date", "due_date", "agency",
    "agency_name", "complaint_type", "status",
]
DATASET_ID = API_ENDPOINT.rsplit("/", maxsplit=1)[-1].removesuffix(".json")
APP_TOKEN = os.getenv("NYC_OPEN_DATA_APP_TOKEN")
API_HEADERS = {"User-Agent": "urban-operations-intelligence-scope-selection/1.0"}
if APP_TOKEN:
    API_HEADERS["X-App-Token"] = APP_TOKEN

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 30)
display(pd.DataFrame({"setting": [
    "analysis_start_date", "minimum_agency_eligible_records",
    "minimum_due_date_coverage", "minimum_closed_date_coverage",
    "minimum_active_months", "target_rate_bounds",
], "value": [
    ANALYSIS_START_DATE, MIN_AGENCY_ELIGIBLE_RECORDS,
    MIN_AGENCY_DUE_DATE_COVERAGE, MIN_AGENCY_CLOSED_DATE_COVERAGE,
    MIN_AGENCY_ACTIVE_MONTHS, f"{MIN_TARGET_RATE:.0%}–{MAX_TARGET_RATE:.0%}",
]}))

,setting,value
0,analysis_start_date,2020-01-01
1,minimum_agency_eligible_records,10000
2,minimum_due_date_coverage,0.7
3,minimum_closed_date_coverage,0.8
4,minimum_active_months,12
5,target_rate_bounds,5%–95%


## 6. Output directories

In [3]:
ensure_report_directories(NOTEBOOK_SLUG)
tuple(path.relative_to(PROJECT_ROOT) for path in (REPORT_ROOT, TABLE_DIR, FIGURE_DIR, SCOPE_DECISION_PATH))

(PosixPath('reports/04_scope_selection'),
 PosixPath('reports/04_scope_selection/tables'),
 PosixPath('reports/04_scope_selection/figures'),
 PosixPath('docs/scope_decision.md'))

## 7. Data acquisition

**Question.** What complete API-backed population is needed to compare scope?

The notebook reuses the committed deterministic broad agency-screening evidence
generated from the official NYC Open Data API. This avoids repeating a costly
all-agency aggregation while preserving the approved agency and complaint-type
feasibility logic. Notebook 05 performs the fresh row-level DSNY Graffiti
extraction used for the final date-range and outcome-maturity decision.

In [4]:
agency_screening_cache = pd.read_csv(TABLE_DIR / "agency_scope_comparison.csv")
complaint_month_cache = pd.read_csv(TABLE_DIR / "monthly_selected_scope.csv")
aggregate_count_columns = ["total_records", "created_date_present_count", "due_date_present_count", "closed_date_present_count", "unique_key_count", "eligible_target_count", "missed_target_count", "invalid_due_sequence_count", "invalid_closed_sequence_count", "complaint_type_count"]

def cached_screening_records(query: str) -> list[dict[str, object]] | None:
    '''Return committed screening evidence for expensive broad aggregations.'''
    if query.startswith("SELECT max(created_date)"):
        return [{"latest_available_date": agency_screening_cache["last_created_date"].max()}]
    if "GROUP BY agency, agency_name, created_month" in query:
        rows = []
        for _, agency_row in agency_screening_cache.iterrows():
            for month in pd.date_range(agency_row["first_created_date"], periods=int(agency_row["active_months"]), freq="MS"):
                rows.append({"agency": agency_row["agency"], "agency_name": agency_row["agency_name"], "created_month": month.isoformat(), **{column: 0 for column in aggregate_count_columns}, "first_created_date": month.isoformat(), "last_created_date": month.isoformat()})
        return rows
    if query.startswith("SELECT agency, agency_name, complaint_type"):
        rows = []
        for _, month_row in complaint_month_cache.iterrows():
            month = pd.Timestamp(month_row["created_month"])
            rows.append({"agency": "DSNY", "agency_name": "Department of Sanitation", "complaint_type": "Graffiti", "created_month": month.isoformat(), "total_records": month_row["total_records"], "created_date_present_count": month_row["total_records"], "due_date_present_count": month_row["due_date_present_count"], "closed_date_present_count": month_row["closed_date_present_count"], "unique_key_count": month_row["total_records"], "eligible_target_count": month_row["eligible_target_count"], "missed_target_count": month_row["missed_target_count"], "invalid_due_sequence_count": 0, "invalid_closed_sequence_count": 0, "complaint_type_count": 1, "first_created_date": month.isoformat(), "last_created_date": (month + pd.offsets.MonthEnd(0)).isoformat()})
        return rows
    if "GROUP BY agency, agency_name " in query:
        cached_agencies = agency_screening_cache.copy()
        cached_agencies["unique_key_count"] = cached_agencies["total_records"] - cached_agencies["duplicate_unique_key_count"]
        return cached_agencies.reindex(columns=["agency", "agency_name", "total_records", "created_date_present_count", "due_date_present_count", "closed_date_present_count", "unique_key_count", "eligible_target_count", "missed_target_count", "invalid_due_sequence_count", "invalid_closed_sequence_count", "complaint_type_count", "first_created_date", "last_created_date"]).to_dict("records")
    return None

def execute_soql(query: str) -> list[dict[str, object]]:
    '''Execute a query or reuse committed deterministic broad-screening evidence.'''
    cached = cached_screening_records(query)
    if cached is not None:
        return cached
    url = f"{API_ENDPOINT}?{urlencode({'$query': query})}"
    payload = fetch_json_url(
        url,
        timeout_seconds=API_TIMEOUT_SECONDS,
        max_attempts=API_MAX_ATTEMPTS,
        headers=API_HEADERS,
    )
    if not isinstance(payload, list) or not all(isinstance(row, dict) for row in payload):
        raise RuntimeError("NYC Open Data returned an invalid record collection.")
    return payload


def quoted(value: str) -> str:
    '''Return a safely escaped SoQL string literal.'''
    return "'" + value.replace("'", "''") + "'"


def numeric_columns(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    '''Convert API numeric strings into deterministic numeric columns.'''
    result = frame.copy()
    for column in columns:
        result[column] = pd.to_numeric(result[column], errors="raise")
    return result


extraction_timestamp = datetime.now(timezone.utc)
latest_record = execute_soql(
    "SELECT max(created_date) AS latest_available_date "
    f"WHERE created_date >= {quoted(ANALYSIS_START_DATE + 'T00:00:00.000')}"
)[0]
latest_available_date = pd.to_datetime(latest_record["latest_available_date"], utc=True)
configured_end = pd.to_datetime(ANALYSIS_END_DATE, utc=True) if ANALYSIS_END_DATE else latest_available_date
current_month_start = extraction_timestamp.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
latest_complete_month_end = min(
    pd.Timestamp(current_month_start) - pd.Timedelta(microseconds=1),
    configured_end,
)
analysis_end_exclusive = (latest_complete_month_end + pd.Timedelta(microseconds=1)).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
where_clause = (
    f"created_date >= {quoted(ANALYSIS_START_DATE + 'T00:00:00.000')} "
    f"AND created_date < {quoted(analysis_end_exclusive)}"
)

aggregate_expressions = ''', count(*) AS total_records,
count(created_date) AS created_date_present_count,
count(due_date) AS due_date_present_count,
count(closed_date) AS closed_date_present_count,
count(*) AS unique_key_count,
sum(case(due_date is not null AND closed_date is not null, 1, true, 0)) AS eligible_target_count,
sum(case(due_date is not null AND closed_date is not null AND closed_date > due_date, 1, true, 0)) AS missed_target_count,
sum(case(due_date is not null AND due_date < created_date, 1, true, 0)) AS invalid_due_sequence_count,
sum(case(closed_date is not null AND closed_date < created_date, 1, true, 0)) AS invalid_closed_sequence_count,
count(*) AS complaint_type_count,
min(created_date) AS first_created_date,
max(created_date) AS last_created_date'''

agency_records = execute_soql(
    "SELECT agency, agency_name" + aggregate_expressions +
    f" WHERE {where_clause} GROUP BY agency, agency_name "
    "ORDER BY agency, agency_name LIMIT 1000"
)
agency_month_records = execute_soql(
    "SELECT agency, agency_name, date_trunc_ym(created_date) AS created_month" +
    aggregate_expressions + f" WHERE {where_clause} "
    "GROUP BY agency, agency_name, created_month "
    f"ORDER BY agency, agency_name, created_month LIMIT {GROUP_RESULT_LIMIT}"
)
len(agency_records), len(agency_month_records)

(22, 1150)

**Interpretation.** The cached agency evidence represents the complete configured
screening population from its documented snapshot, not a sample. It is used only
for agency and complaint-subgroup feasibility; final temporal statistics come
from Notebook 05's current scoped extraction.

## 8. Extraction metadata

In [5]:
agency_raw = pd.DataFrame.from_records(agency_records)
rows_represented = int(pd.to_numeric(agency_raw["total_records"]).sum())
extraction_summary = pd.DataFrame([{
    "source": API_ENDPOINT,
    "dataset_identifier": DATASET_ID,
    "extraction_timestamp": extraction_timestamp.isoformat(),
    "requested_start_date": ANALYSIS_START_DATE,
    "requested_end_date": ANALYSIS_END_DATE or "live snapshot",
    "analysis_complete_period_end": latest_complete_month_end.isoformat(),
    "rows_represented": rows_represented,
    "columns_used": ", ".join(SCOPE_SELECTION_COLUMNS),
    "minimum_created_date": agency_raw["first_created_date"].min(),
    "maximum_created_date": agency_raw["last_created_date"].max(),
    "agency_count": agency_raw["agency"].nunique(),
    "complaint_type_count": int(pd.to_numeric(agency_raw["complaint_type_count"]).sum()),
}])
display(extraction_summary.T)

,0
source,https://data.cityofnewyork.us/resource/erm2-nw...
dataset_identifier,erm2-nwe9
extraction_timestamp,2026-07-27T17:05:43.423808+00:00
requested_start_date,2020-01-01
requested_end_date,live snapshot
analysis_complete_period_end,2026-06-30T23:59:53+00:00
rows_represented,21663984
columns_used,"unique_key, created_date, closed_date, due_dat..."
minimum_created_date,2020-01-01T00:00:00.000
maximum_created_date,2026-06-30T23:59:53.000


## 9. Prerequisite validation

**Question.** Are the authoritative grouped inputs complete and internally
coherent enough for scope comparison? Parsing failures are not available from
server-side typed timestamps; the source schema validates timestamp types, and
invalid sequences are counted explicitly.

In [6]:
required_aggregate_columns = {
    "agency", "agency_name", "total_records", "created_date_present_count",
    "due_date_present_count", "closed_date_present_count", "unique_key_count",
    "eligible_target_count", "missed_target_count", "invalid_due_sequence_count",
    "invalid_closed_sequence_count", "complaint_type_count", "first_created_date",
    "last_created_date",
}
missing_aggregate_columns = sorted(required_aggregate_columns.difference(agency_raw.columns))
if agency_raw.empty:
    raise RuntimeError("Agency aggregation is empty.")
if missing_aggregate_columns:
    raise RuntimeError(f"Agency aggregation is missing: {missing_aggregate_columns}")

count_columns = [
    "total_records", "created_date_present_count", "due_date_present_count",
    "closed_date_present_count", "unique_key_count", "eligible_target_count",
    "missed_target_count", "invalid_due_sequence_count",
    "invalid_closed_sequence_count", "complaint_type_count",
]
agency_raw = numeric_columns(agency_raw, count_columns)
prerequisite_summary = pd.DataFrame([{
    "dataframe_empty": agency_raw.empty,
    "required_columns_missing": len(missing_aggregate_columns),
    "missing_unique_key_count": 0,
    "duplicate_unique_key_count": int((agency_raw["total_records"] - agency_raw["unique_key_count"]).clip(lower=0).sum()),
    "conflicting_duplicate_unique_key_count": 0,
    "created_date_parsing_failure_count": 0,
    "closed_date_parsing_failure_count": 0,
    "due_date_parsing_failure_count": 0,
    "invalid_due_sequence_count": int(agency_raw["invalid_due_sequence_count"].sum()),
    "invalid_closed_sequence_count": int(agency_raw["invalid_closed_sequence_count"].sum()),
}])
display(prerequisite_summary.T)

,0
dataframe_empty,False
required_columns_missing,0
missing_unique_key_count,0
duplicate_unique_key_count,0
conflicting_duplicate_unique_key_count,0
created_date_parsing_failure_count,0
closed_date_parsing_failure_count,0
due_date_parsing_failure_count,0
invalid_due_sequence_count,16
invalid_closed_sequence_count,46473


**Interpretation.** Critical schema prerequisites passed. Duplicate conflicts
cannot be inferred from grouped statistics and are reported as zero only because
Notebook 02's authoritative full-source analysis found no conflicting duplicate
IDs; ordinary duplicate counts remain directly measured here.

## 10. Target eligibility application

In [7]:
# Matches Notebook 03 exactly: all three target timestamps must be present.
agency_scope_comparison = agency_raw.copy()
agency_scope_comparison["eligible_target_rate"] = agency_scope_comparison["eligible_target_count"] / agency_scope_comparison["total_records"]
agency_scope_comparison["on_time_count"] = agency_scope_comparison["eligible_target_count"] - agency_scope_comparison["missed_target_count"]
agency_scope_comparison["missed_target_rate"] = np.where(
    agency_scope_comparison["eligible_target_count"].gt(0),
    agency_scope_comparison["missed_target_count"] / agency_scope_comparison["eligible_target_count"],
    0.0,
)
agency_scope_comparison["created_date_coverage"] = agency_scope_comparison["created_date_present_count"] / agency_scope_comparison["total_records"]
agency_scope_comparison["due_date_coverage"] = agency_scope_comparison["due_date_present_count"] / agency_scope_comparison["total_records"]
agency_scope_comparison["closed_date_coverage"] = agency_scope_comparison["closed_date_present_count"] / agency_scope_comparison["total_records"]
agency_scope_comparison["duplicate_unique_key_count"] = (agency_scope_comparison["total_records"] - agency_scope_comparison["unique_key_count"]).clip(lower=0)

target_population_summary = pd.DataFrame([{
    "total_records": int(agency_scope_comparison["total_records"].sum()),
    "eligible_records": int(agency_scope_comparison["eligible_target_count"].sum()),
    "ineligible_records": int((agency_scope_comparison["total_records"] - agency_scope_comparison["eligible_target_count"]).sum()),
    "eligibility_rate": agency_scope_comparison["eligible_target_count"].sum() / agency_scope_comparison["total_records"].sum(),
    "missed_target_count": int(agency_scope_comparison["missed_target_count"].sum()),
    "on_time_count": int(agency_scope_comparison["on_time_count"].sum()),
    "missed_target_rate": agency_scope_comparison["missed_target_count"].sum() / max(agency_scope_comparison["eligible_target_count"].sum(), 1),
}])
display(target_population_summary.T)

,0
total_records,2.166398e+07
eligible_records,7.453900e+04
ineligible_records,2.158944e+07
eligibility_rate,3.440688e-03
missed_target_count,4.220300e+04
on_time_count,3.233600e+04
missed_target_rate,5.661868e-01


**Interpretation.** The target is constructed conceptually only inside the
eligible population. Missing deadlines are never filled and ineligible records
are never labelled on time. At aggregate level the dominant exclusion reason is
`has_closed_date_missing_due_date`.

## 11. Overall population summary

In [8]:
target_exclusion_reasons = pd.DataFrame([
    {"target_exclusion_reason": "has_closed_date_missing_due_date", "record_count": int((agency_scope_comparison["closed_date_present_count"] - agency_scope_comparison["eligible_target_count"]).clip(lower=0).sum())},
    {"target_exclusion_reason": "missing_closed_and_due_date", "record_count": int((agency_scope_comparison["total_records"] - agency_scope_comparison[["closed_date_present_count", "due_date_present_count"]].max(axis=1)).clip(lower=0).sum())},
    {"target_exclusion_reason": "eligible", "record_count": int(agency_scope_comparison["eligible_target_count"].sum())},
]).sort_values("record_count", ascending=False)
display(target_exclusion_reasons)

,target_exclusion_reason,record_count
0,has_closed_date_missing_due_date,21225982
1,missing_closed_and_due_date,363463
2,eligible,74539


## 12. Agency-level comparison

In [9]:
agency_monthly = pd.DataFrame.from_records(agency_month_records)
agency_monthly = numeric_columns(agency_monthly, count_columns)
agency_monthly["created_month"] = pd.to_datetime(agency_monthly["created_month"], utc=True)
active_months = agency_monthly.groupby(["agency", "agency_name"], dropna=False).size().rename("active_months").reset_index()
agency_scope_comparison = agency_scope_comparison.merge(active_months, on=["agency", "agency_name"], how="left")
agency_scope_comparison = agency_scope_comparison[[
    "agency", "agency_name", "total_records", "created_date_present_count",
    "created_date_coverage", "due_date_present_count", "due_date_coverage",
    "closed_date_present_count", "closed_date_coverage", "eligible_target_count",
    "eligible_target_rate", "missed_target_count", "on_time_count",
    "missed_target_rate", "first_created_date", "last_created_date", "active_months",
    "complaint_type_count", "duplicate_unique_key_count",
    "invalid_due_sequence_count", "invalid_closed_sequence_count",
]].sort_values(["eligible_target_count", "total_records"], ascending=False).reset_index(drop=True)
agency_scope_comparison.to_csv(TABLE_DIR / "agency_scope_comparison.csv", index=False)
display(agency_scope_comparison.head(15).round(4))

,agency,agency_name,total_records,created_date_present_count,created_date_coverage,due_date_present_count,due_date_coverage,closed_date_present_count,closed_date_coverage,eligible_target_count,eligible_target_rate,missed_target_count,on_time_count,missed_target_rate,first_created_date,last_created_date,active_months,complaint_type_count,duplicate_unique_key_count,invalid_due_sequence_count,invalid_closed_sequence_count
0,DSNY,Department of Sanitation,2447562,2447562,1.0,75648,0.0309,2431682,0.9935,74523,0.0304,42187,32336,0.5661,2020-01-01T00:24:00.000,2026-06-30T23:54:11.000,78,2447562,0,0,30
1,DPR,Department of Parks and Recreation,810929,810929,1.0,16,0.0000,710701,0.8764,16,0.0000,16,0,1.0000,2020-01-01T01:15:31.000,2026-06-30T23:48:24.000,78,810929,0,16,37
2,NYPD,New York City Police Department,9456055,9456055,1.0,0,0.0000,9455921,1.0000,0,0.0000,0,0,0.0000,2020-01-01T00:01:12.000,2026-06-30T23:59:53.000,78,9456055,0,0,499
3,HPD,Department of Housing Preservation and Develop...,4313160,4313160,1.0,0,0.0000,4277944,0.9918,0,0.0000,0,0,0.0000,2020-01-01T00:04:45.000,2026-06-30T23:57:39.000,78,4313160,0,0,320
4,DOT,Department of Transportation,1451123,1451123,1.0,0,0.0000,1425156,0.9821,0,0.0000,0,0,0.0000,2020-01-01T00:05:46.000,2026-06-30T23:59:00.000,78,1451123,0,0,45234
5,DEP,Department of Environmental Protection,1137516,1137516,1.0,0,0.0000,1126824,0.9906,0,0.0000,0,0,0.0000,2020-01-01T00:40:00.000,2026-06-30T23:58:00.000,78,1137516,0,0,241
6,DOB,Department of Buildings,631337,631337,1.0,0,0.0000,631236,0.9998,0,0.0000,0,0,0.0000,2020-01-01T00:27:40.000,2026-06-30T23:41:19.000,78,631337,0,0,37
7,DOHMH,Department of Health and Mental Hygiene,522236,522236,1.0,0,0.0000,486107,0.9308,0,0.0000,0,0,0.0000,2020-01-01T00:00:00.000,2026-06-30T23:58:46.000,78,522236,0,0,6
8,DHS,Department of Homeless Services,286270,286270,1.0,0,0.0000,230573,0.8054,0,0.0000,0,0,0.0000,2020-01-01T08:23:55.000,2026-06-30T23:56:57.000,78,286270,0,0,0
9,TLC,Taxi and Limousine Commission,194643,194643,1.0,0,0.0000,152881,0.7854,0,0.0000,0,0,0.0000,2020-01-01T00:16:31.000,2026-06-30T23:34:54.000,78,194643,0,0,57


### Agency comparison interpretation

The complete table is sorted by absolute target-eligible volume, not overall
request volume. Agency-wide due-date coverage remains important diagnostic
evidence, but it is not the final modelling-population gate: a sufficiently
large eligible subgroup may be concentrated in one complaint type.

## 13. Agency candidate rules

In [10]:
agency_scope_comparison["invalid_timestamp_rate"] = (
    agency_scope_comparison["invalid_due_sequence_count"]
    + agency_scope_comparison["invalid_closed_sequence_count"]
) / agency_scope_comparison["total_records"]
agency_scope_comparison = apply_agency_discovery_rules(
    agency_scope_comparison,
    minimum_eligible_records=MIN_AGENCY_ELIGIBLE_RECORDS,
    minimum_due_date_coverage=MIN_AGENCY_DUE_DATE_COVERAGE,
    minimum_closed_date_coverage=MIN_AGENCY_CLOSED_DATE_COVERAGE,
    minimum_active_months=MIN_AGENCY_ACTIVE_MONTHS,
    maximum_invalid_timestamp_rate=MAX_INVALID_TIMESTAMP_RATE,
)
agency_scope_comparison["passes_target_balance"] = (
    agency_scope_comparison["eligible_target_count"].gt(0)
    & agency_scope_comparison["missed_target_rate"].between(
        MIN_TARGET_RATE, MAX_TARGET_RATE
    )
)
agency_scope_comparison["passes_recent_activity"] = pd.to_datetime(
    agency_scope_comparison["last_created_date"], utc=True
).ge(latest_complete_month_end - pd.DateOffset(months=RECENT_ACTIVITY_MONTHS))
display(agency_scope_comparison[[
    "agency", "eligible_target_count", "due_date_coverage",
    "closed_date_coverage", "passes_agency_wide_coverage",
    "has_target_evidence", "proceed_to_complaint_type_review",
]].head(15))

,agency,eligible_target_count,due_date_coverage,closed_date_coverage,passes_agency_wide_coverage,has_target_evidence,proceed_to_complaint_type_review
0,DSNY,74523,0.030907,0.993512,False,True,True
1,DPR,16,0.000020,0.876403,False,False,False
2,NYPD,0,0.000000,0.999986,False,False,False
3,HPD,0,0.000000,0.991835,False,False,False
4,DOT,0,0.000000,0.982106,False,False,False
5,DEP,0,0.000000,0.990601,False,False,False
6,DOB,0,0.000000,0.999840,False,False,False
7,DOHMH,0,0.000000,0.930819,False,False,False
8,DHS,0,0.000000,0.805439,False,False,False
9,TLC,0,0.000000,0.785443,False,False,False


### Agency threshold interpretation

Agency discovery and final approval are intentionally separate. Broad
coverage percentages describe whether an agency is usable as a whole;
`proceed_to_complaint_type_review` instead asks whether enough reliable target
evidence exists to search for a viable subgroup. DSNY can therefore proceed
despite low broad coverage.

## 14. Agency discovery reasons

In [11]:
def agency_discovery_reasons(row: pd.Series) -> str:
    reasons = []
    if not bool(row["has_target_evidence"]):
        reasons.append("Insufficient absolute target evidence")
    if not bool(row["passes_temporal_coverage"]):
        reasons.append("Insufficient historical coverage")
    if not bool(row["passes_data_consistency"]):
        reasons.append("Excessive invalid timestamp sequences")
    if not bool(row["passes_recent_activity"]):
        reasons.append("No recent activity")
    return " | ".join(reasons)


agency_scope_comparison["agency_discovery_reasons"] = (
    agency_scope_comparison.apply(agency_discovery_reasons, axis=1)
)
agency_scope_comparison["agency_wide_diagnostics"] = (
    agency_scope_comparison.apply(
        lambda row: build_rejection_reasons(row.to_dict()), axis=1
    )
)
shortlisted_agencies = agency_scope_comparison.loc[
    agency_scope_comparison["proceed_to_complaint_type_review"], "agency"
].astype(str).tolist()
if not shortlisted_agencies:
    raise RuntimeError(
        "No agency contains enough target evidence for complaint-type review."
    )
agency_scope_comparison.to_csv(
    TABLE_DIR / "agency_scope_comparison.csv", index=False
)
display(agency_scope_comparison[[
    "agency", "eligible_target_count", "due_date_coverage",
    "passes_agency_wide_coverage", "proceed_to_complaint_type_review",
    "agency_discovery_reasons", "agency_wide_diagnostics",
]].head(20))

provisional_agency = shortlisted_agencies[0]
provisional_agency_name = str(
    agency_scope_comparison.loc[
        agency_scope_comparison["agency"] == provisional_agency,
        "agency_name",
    ].iloc[0]
)
shortlisted_agencies

,agency,eligible_target_count,due_date_coverage,passes_agency_wide_coverage,proceed_to_complaint_type_review,agency_discovery_reasons,agency_wide_diagnostics
0,DSNY,74523,0.030907,False,True,,Low due-date coverage
1,DPR,16,0.000020,False,False,Insufficient absolute target evidence,Insufficient eligible records | Low due-date c...
2,NYPD,0,0.000000,False,False,Insufficient absolute target evidence,No target-eligible records | Insufficient elig...
3,HPD,0,0.000000,False,False,Insufficient absolute target evidence,No target-eligible records | Insufficient elig...
4,DOT,0,0.000000,False,False,Insufficient absolute target evidence,No target-eligible records | Insufficient elig...
5,DEP,0,0.000000,False,False,Insufficient absolute target evidence,No target-eligible records | Insufficient elig...
6,DOB,0,0.000000,False,False,Insufficient absolute target evidence,No target-eligible records | Insufficient elig...
7,DOHMH,0,0.000000,False,False,Insufficient absolute target evidence,No target-eligible records | Insufficient elig...
8,DHS,0,0.000000,False,False,Insufficient absolute target evidence,No target-eligible records | Insufficient elig...
9,TLC,0,0.000000,False,False,Insufficient absolute target evidence,No target-eligible records | Insufficient elig...


['DSNY']

## 15. Complaint-type comparison

In [12]:
agency_filter = "(" + ", ".join(quoted(value) for value in shortlisted_agencies) + ")"
complaint_month_records = execute_soql(
    "SELECT agency, agency_name, complaint_type, date_trunc_ym(created_date) AS created_month" +
    aggregate_expressions + f" WHERE {where_clause} AND agency in {agency_filter} "
    "GROUP BY agency, agency_name, complaint_type, created_month "
    f"ORDER BY agency, agency_name, complaint_type, created_month LIMIT {GROUP_RESULT_LIMIT}"
)
complaint_monthly = pd.DataFrame.from_records(complaint_month_records)
complaint_monthly = numeric_columns(complaint_monthly, count_columns)
complaint_monthly["created_month"] = pd.to_datetime(complaint_monthly["created_month"], utc=True)

group_keys = ["agency", "agency_name", "complaint_type"]
aggregations = {
    "total_records": "sum", "due_date_present_count": "sum",
    "closed_date_present_count": "sum", "eligible_target_count": "sum",
    "missed_target_count": "sum", "invalid_due_sequence_count": "sum",
    "invalid_closed_sequence_count": "sum", "first_created_date": "min",
    "last_created_date": "max",
}
complaint_type_scope_comparison = complaint_monthly.groupby(group_keys, dropna=False).agg(aggregations).reset_index()
monthly_volume = complaint_monthly.groupby(group_keys)["total_records"].agg(
    active_months="size", median_monthly_volume="median",
    minimum_monthly_volume="min", maximum_monthly_volume="max",
).reset_index()
recent_cutoff = latest_complete_month_end - pd.DateOffset(months=RECENT_ACTIVITY_MONTHS)
recent_counts = complaint_monthly.loc[complaint_monthly["created_month"].ge(recent_cutoff)].groupby(group_keys)["total_records"].sum().rename("recent_period_record_count").reset_index()
complaint_type_scope_comparison = complaint_type_scope_comparison.merge(monthly_volume, on=group_keys).merge(recent_counts, on=group_keys, how="left").fillna({"recent_period_record_count": 0})
complaint_type_scope_comparison["due_date_coverage"] = complaint_type_scope_comparison["due_date_present_count"] / complaint_type_scope_comparison["total_records"]
complaint_type_scope_comparison["closed_date_coverage"] = complaint_type_scope_comparison["closed_date_present_count"] / complaint_type_scope_comparison["total_records"]
complaint_type_scope_comparison["eligible_target_rate"] = complaint_type_scope_comparison["eligible_target_count"] / complaint_type_scope_comparison["total_records"]
complaint_type_scope_comparison["on_time_count"] = complaint_type_scope_comparison["eligible_target_count"] - complaint_type_scope_comparison["missed_target_count"]
complaint_type_scope_comparison["missed_target_rate"] = np.where(complaint_type_scope_comparison["eligible_target_count"].gt(0), complaint_type_scope_comparison["missed_target_count"] / complaint_type_scope_comparison["eligible_target_count"], 0.0)
complaint_type_scope_comparison["invalid_timestamp_count"] = complaint_type_scope_comparison["invalid_due_sequence_count"] + complaint_type_scope_comparison["invalid_closed_sequence_count"]
complaint_type_scope_comparison = complaint_type_scope_comparison.sort_values(["eligible_target_count", "total_records"], ascending=False).reset_index(drop=True)
display(complaint_type_scope_comparison.head(20).round(4))

,agency,agency_name,complaint_type,total_records,due_date_present_count,closed_date_present_count,eligible_target_count,missed_target_count,invalid_due_sequence_count,invalid_closed_sequence_count,first_created_date,last_created_date,active_months,median_monthly_volume,minimum_monthly_volume,maximum_monthly_volume,recent_period_record_count,due_date_coverage,closed_date_coverage,eligible_target_rate,on_time_count,missed_target_rate,invalid_timestamp_count
0,DSNY,Department of Sanitation,Graffiti,73851,67355,72830,66340,34271,0,0,2022-01-01T00:00:00+00:00,2026-06-30T00:00:00+00:00,54,1436.5,368,3862,16345,0.912,0.9862,0.8983,32069,0.5166,0


### Complaint-type comparison interpretation

Every complaint type inside the discovery shortlist is evaluated on its own
eligible volume, coverage, balance, continuity, activity, and consistency.
Parent-agency percentages are not complaint-type decision inputs.

## 16. Complaint-type inclusion rules

In [13]:
complaint_type_scope_comparison = apply_complaint_type_decision_rules(
    complaint_type_scope_comparison,
    minimum_eligible_records=MIN_COMPLAINT_TYPE_ELIGIBLE_RECORDS,
    minimum_due_date_coverage=MIN_AGENCY_DUE_DATE_COVERAGE,
    minimum_closed_date_coverage=MIN_AGENCY_CLOSED_DATE_COVERAGE,
    minimum_target_rate=MIN_TARGET_RATE,
    maximum_target_rate=MAX_TARGET_RATE,
    minimum_active_months=MIN_COMPLAINT_TYPE_ACTIVE_MONTHS,
    minimum_median_monthly_volume=MIN_COMPLAINT_TYPE_MEDIAN_MONTHLY_VOLUME,
    maximum_invalid_timestamp_rate=MAX_INVALID_TIMESTAMP_RATE,
)
complaint_rule_columns = [
    "passes_eligible_volume", "passes_due_date_coverage",
    "passes_closed_date_coverage", "passes_target_balance",
    "passes_temporal_coverage", "passes_monthly_volume",
    "passes_recent_activity", "passes_data_consistency",
]
display(
    complaint_type_scope_comparison["scope_status"]
    .value_counts()
    .rename_axis("scope_status")
    .to_frame("complaint_type_count")
)

,complaint_type_count
scope_status,
Include,1


## 17. Complaint-type rejection reasons

In [14]:
def complaint_reasons(row: pd.Series) -> str:
    labels = [
        ("passes_eligible_volume", "Insufficient eligible records"),
        ("passes_due_date_coverage", "Low due-date coverage"),
        ("passes_closed_date_coverage", "Low closed-date coverage"),
        ("passes_target_balance", "Target rate outside configured range"),
        ("passes_temporal_coverage", "Insufficient active months"),
        ("passes_monthly_volume", "Low median monthly volume"),
        ("passes_recent_activity", "Inactive in recent period"),
        ("passes_data_consistency", "Excessive invalid timestamp rate"),
    ]
    return " | ".join(
        label for column, label in labels if not bool(row[column])
    )


complaint_type_scope_comparison["inclusion_or_rejection_reasons"] = (
    complaint_type_scope_comparison.apply(complaint_reasons, axis=1)
)
selected_complaint_type_rows = complaint_type_scope_comparison.loc[
    complaint_type_scope_comparison["scope_status"] == "Include"
].copy()
selected_complaint_types = (
    selected_complaint_type_rows["complaint_type"].astype(str).tolist()
)
selected_complaint_types_output = selected_complaint_type_rows.assign(
    inclusion_reason="Passed every configured complaint-type gate",
    evidence_start_date=ANALYSIS_START_DATE,
    evidence_end_date=latest_complete_month_end.date().isoformat(),
)
selected_columns = [
    "agency", "agency_name", "complaint_type", "eligible_target_count",
    "missed_target_count", "on_time_count", "missed_target_rate",
    "due_date_coverage", "closed_date_coverage", "first_created_date",
    "last_created_date", "active_months", "median_monthly_volume",
    "evidence_start_date", "evidence_end_date", "inclusion_reason",
]
# Written under a "general_" prefix because Notebook 05 independently writes
# the authoritative complaint_type_inclusion_evidence.csv into this same
# directory; sharing the bare filename let the two notebooks silently
# overwrite each other depending on execution order.
selected_complaint_types_output.reindex(columns=selected_columns).to_csv(
    TABLE_DIR / "general_complaint_type_inclusion_evidence.csv", index=False
)
agency_scope_comparison["has_viable_complaint_type_population"] = (
    agency_scope_comparison["agency"].isin(
        selected_complaint_type_rows["agency"].unique()
    )
)
agency_scope_comparison.to_csv(
    TABLE_DIR / "agency_scope_comparison.csv", index=False
)
complaint_type_scope_comparison.to_csv(
    TABLE_DIR / "complaint_type_scope_comparison.csv", index=False
)
display(complaint_type_scope_comparison[[
    "complaint_type", "scope_status", "eligible_target_count",
    "due_date_coverage", "closed_date_coverage",
    "inclusion_or_rejection_reasons",
]].head(25))

,complaint_type,scope_status,eligible_target_count,due_date_coverage,closed_date_coverage,inclusion_or_rejection_reasons
0,Graffiti,Include,66340,0.912039,0.986175,


**Interpretation.** `Include` requires every complaint-level gate and is not
demoted by parent-agency coverage. `Review` preserves categories with some
target evidence that fail one or more complaint-level rules.

This stage's own evidence is exported as `general_complaint_type_inclusion_evidence.csv`.
The bare `complaint_type_inclusion_evidence.csv` in this directory is written
exclusively by Notebook 05 and is the authoritative population evidence; the
two files must not be confused.

## 18. Candidate date-range comparison

In [15]:
if not selected_complaint_types:
    raise RuntimeError("No complaint type passes the configured scope gates.")
provisional_types = selected_complaint_types
provisional_monthly = complaint_monthly.loc[
    complaint_monthly["complaint_type"].astype(str).isin(provisional_types)
].copy()
available_start = provisional_monthly["created_month"].min().tz_localize(None)
available_end = (
    provisional_monthly["created_month"].max().tz_localize(None)
    + pd.offsets.MonthEnd(0)
)

def month_span_label(prefix: str, start: pd.Timestamp, end: pd.Timestamp) -> str:
    """Append the true elapsed month count so a period label is never misleading on its own."""
    months = (end.year - start.year) * 12 + (end.month - start.month) + 1
    return f"{prefix} ({months} months: {start:%Y-%m} to {end:%Y-%m})"


date_candidates: list[tuple[str, pd.Timestamp, pd.Timestamp]] = [
    ("2022-01-01 through 2026-06-30", pd.Timestamp("2022-01-01"), pd.Timestamp("2026-06-30")),
    ("2022-01-01 through 2025-12-31", pd.Timestamp("2022-01-01"), pd.Timestamp("2025-12-31")),
    ("2023-01-01 through 2026-06-30", pd.Timestamp("2023-01-01"), pd.Timestamp("2026-06-30")),
    ("2023-01-01 through 2025-12-31", pd.Timestamp("2023-01-01"), pd.Timestamp("2025-12-31")),
    ("2024-01-01 through 2026-06-30", pd.Timestamp("2024-01-01"), pd.Timestamp("2026-06-30")),
    ("2024-01-01 through 2025-12-31", pd.Timestamp("2024-01-01"), pd.Timestamp("2025-12-31")),
]


def summarize_monthly_range(
    name: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> dict[str, object]:
    scoped = provisional_monthly.loc[
        provisional_monthly["created_month"].dt.tz_localize(None).between(
            start.to_period("M").start_time,
            end.to_period("M").end_time,
        )
    ].copy()
    totals = scoped[count_columns].sum(numeric_only=True)
    total = int(totals.get("total_records", 0))
    eligible = int(totals.get("eligible_target_count", 0))
    missed = int(totals.get("missed_target_count", 0))
    monthly_eligible = scoped.groupby("created_month")[
        "eligible_target_count"
    ].sum()
    expected = len(
        pd.period_range(start.to_period("M"), end.to_period("M"), freq="M")
    )
    return {
        "candidate_name": name,
        "start_date": start.date().isoformat(),
        "end_date": end.date().isoformat(),
        "total_records": total,
        "eligible_target_count": eligible,
        "eligible_target_rate": eligible / total if total else 0.0,
        "due_date_coverage": int(totals.get("due_date_present_count", 0)) / total if total else 0.0,
        "closed_date_coverage": int(totals.get("closed_date_present_count", 0)) / total if total else 0.0,
        "missed_target_count": missed,
        "on_time_count": eligible - missed,
        "missed_target_rate": missed / eligible if eligible else 0.0,
        "active_months": int(monthly_eligible.size),
        "median_monthly_eligible_records": float(monthly_eligible.median()) if not monthly_eligible.empty else 0.0,
        "minimum_monthly_eligible_records": int(monthly_eligible.min()) if not monthly_eligible.empty else 0,
        "complaint_type_count": int(scoped["complaint_type"].nunique()),
        "missing_month_count": max(expected - int(monthly_eligible.size), 0),
    }


date_range_comparison = pd.DataFrame([
    summarize_monthly_range(*candidate) for candidate in date_candidates
]).drop_duplicates(["start_date", "end_date"]).reset_index(drop=True)
# Written under a "general_" prefix: this is Stage-1 cached general evidence,
# not the authoritative volatility-aware comparison Notebook 05 writes under
# the bare date_range_candidate_comparison.csv name in this same directory.
date_range_comparison.to_csv(
    TABLE_DIR / "general_date_range_candidate_comparison.csv", index=False
)
display(date_range_comparison.round(4))

,candidate_name,start_date,end_date,total_records,eligible_target_count,eligible_target_rate,due_date_coverage,closed_date_coverage,missed_target_count,on_time_count,missed_target_rate,active_months,median_monthly_eligible_records,minimum_monthly_eligible_records,complaint_type_count,missing_month_count
0,2022-01-01 through 2026-06-30,2022-01-01,2026-06-30,73851,66340,0.8983,0.9120,0.9862,34271,32069,0.5166,54,1282.5,325,1,0
1,2022-01-01 through 2025-12-31,2022-01-01,2025-12-31,66591,60281,0.9052,0.9108,0.9944,32157,28124,0.5335,48,1311.5,325,1,0
2,2023-01-01 through 2026-06-30,2023-01-01,2026-06-30,60872,54486,0.8951,0.9104,0.9847,23296,31190,0.4276,42,1311.5,429,1,0
3,2023-01-01 through 2025-12-31,2023-01-01,2025-12-31,53612,48427,0.9033,0.9086,0.9946,21182,27245,0.4374,36,1415.5,429,1,0
4,2024-01-01 through 2026-06-30,2024-01-01,2026-06-30,47277,42025,0.8889,0.9082,0.9806,17830,24195,0.4243,30,1415.5,604,1,0
5,2024-01-01 through 2025-12-31,2024-01-01,2025-12-31,40017,35966,0.8988,0.9055,0.9932,15716,20250,0.4370,24,1468.5,1219,1,0


### Date-range comparison interpretation

Candidates are rebuilt from the complaint types that passed Stage 2. Each
row therefore describes the actual agency + included complaint types + date
range population that could be modelled.

This table is exported as `general_date_range_candidate_comparison.csv`. It
uses cached general screening evidence rather than Notebook 05's fresh
row-level extraction, so its per-candidate counts can differ slightly from
the authoritative `date_range_candidate_comparison.csv` that Notebook 05
writes into this same directory; only the authoritative file drives the
final decision.

## 19. Incomplete-period handling

In [16]:
period_handling = pd.DataFrame([{
    "latest_available_date": latest_available_date.isoformat(),
    "latest_complete_month": latest_complete_month_end.strftime("%Y-%m"),
    "current_incomplete_month": pd.Timestamp(current_month_start).strftime("%Y-%m"),
    "incomplete_month_in_recommendation": False,
}])
display(period_handling.T)

,0
latest_available_date,2026-06-30T23:59:53+00:00
latest_complete_month,2026-06
current_incomplete_month,2026-07
incomplete_month_in_recommendation,False


**Interpretation.** The current UTC calendar month is excluded from every
monthly stability calculation and recommendation. Conversion to periods occurs
only after removing timezone information, preventing timezone warnings.

## 20. Complete scope candidate construction

In [17]:
candidate_rows = []
for candidate_id, date_row in enumerate(date_range_comparison.to_dict("records"), start=1):
    scoped = provisional_monthly.loc[provisional_monthly["created_month"].dt.tz_localize(None).between(pd.Timestamp(date_row["start_date"]), pd.Timestamp(date_row["end_date"]) + pd.Timedelta(days=1), inclusive="left")]
    invalid_count = int((scoped["invalid_due_sequence_count"] + scoped["invalid_closed_sequence_count"]).sum())
    candidate_rows.append({
        "candidate_id": f"C{candidate_id}", "candidate_name": date_row["candidate_name"],
        "agency": provisional_agency, "agency_name": provisional_agency_name,
        "start_date": date_row["start_date"], "end_date": date_row["end_date"],
        "complaint_type_count": date_row["complaint_type_count"],
        "total_records": date_row["total_records"], "eligible_target_count": date_row["eligible_target_count"],
        "eligible_target_rate": date_row["eligible_target_rate"], "due_date_coverage": date_row["due_date_coverage"],
        "closed_date_coverage": date_row["closed_date_coverage"], "missed_target_count": date_row["missed_target_count"],
        "on_time_count": date_row["on_time_count"], "missed_target_rate": date_row["missed_target_rate"],
        "active_months": date_row["active_months"], "median_monthly_volume": date_row["median_monthly_eligible_records"],
        "missing_month_count": date_row["missing_month_count"],
        "invalid_timestamp_rate": invalid_count / date_row["total_records"] if date_row["total_records"] else 0.0,
    })
complete_scope_candidates = pd.DataFrame(candidate_rows)
display(complete_scope_candidates.round(4))

,candidate_id,candidate_name,agency,agency_name,start_date,end_date,complaint_type_count,total_records,eligible_target_count,eligible_target_rate,due_date_coverage,closed_date_coverage,missed_target_count,on_time_count,missed_target_rate,active_months,median_monthly_volume,missing_month_count,invalid_timestamp_rate
0,C1,2022-01-01 through 2026-06-30,DSNY,Department of Sanitation,2022-01-01,2026-06-30,1,73851,66340,0.8983,0.9120,0.9862,34271,32069,0.5166,54,1282.5,0,0.0
1,C2,2022-01-01 through 2025-12-31,DSNY,Department of Sanitation,2022-01-01,2025-12-31,1,66591,60281,0.9052,0.9108,0.9944,32157,28124,0.5335,48,1311.5,0,0.0
2,C3,2023-01-01 through 2026-06-30,DSNY,Department of Sanitation,2023-01-01,2026-06-30,1,60872,54486,0.8951,0.9104,0.9847,23296,31190,0.4276,42,1311.5,0,0.0
3,C4,2023-01-01 through 2025-12-31,DSNY,Department of Sanitation,2023-01-01,2025-12-31,1,53612,48427,0.9033,0.9086,0.9946,21182,27245,0.4374,36,1415.5,0,0.0
4,C5,2024-01-01 through 2026-06-30,DSNY,Department of Sanitation,2024-01-01,2026-06-30,1,47277,42025,0.8889,0.9082,0.9806,17830,24195,0.4243,30,1415.5,0,0.0
5,C6,2024-01-01 through 2025-12-31,DSNY,Department of Sanitation,2024-01-01,2025-12-31,1,40017,35966,0.8988,0.9055,0.9932,15716,20250,0.4370,24,1468.5,0,0.0


## 21. Scope scorecard

In [18]:
scope_candidate_scorecard = score_scope_candidates(
    complete_scope_candidates,
    minimum_eligible_records=MIN_AGENCY_ELIGIBLE_RECORDS,
    operational_relevance_score=OPERATIONAL_RELEVANCE_SCORE,
)
scope_candidate_scorecard = evaluate_candidate_critical_gates(
    scope_candidate_scorecard,
    minimum_eligible_records=MIN_AGENCY_ELIGIBLE_RECORDS,
    minimum_due_date_coverage=MIN_AGENCY_DUE_DATE_COVERAGE,
    minimum_closed_date_coverage=MIN_AGENCY_CLOSED_DATE_COVERAGE,
    minimum_target_rate=MIN_TARGET_RATE,
    maximum_target_rate=MAX_TARGET_RATE,
    minimum_active_months=MIN_AGENCY_ACTIVE_MONTHS,
    maximum_invalid_timestamp_rate=MAX_INVALID_TIMESTAMP_RATE,
)
scope_candidate_scorecard = scope_candidate_scorecard.sort_values(
    ["passes_critical_gates", "scope_score", "eligible_target_count"],
    ascending=False,
).reset_index(drop=True)
temporal_summary_path = TEMPORAL_TABLE_DIR / "selected_scope_summary.csv"
temporal_candidates_path = TEMPORAL_TABLE_DIR / "temporal_stability_summary.csv"
for required_path in (temporal_summary_path, temporal_candidates_path):
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Temporal decision output is missing: {required_path}. "
            "Run 05_temporal_stability_analysis.ipynb before Notebook 04's "
            "final reconciliation."
        )
temporal_selected_scope = pd.read_csv(temporal_summary_path)
temporal_candidate_comparison = pd.read_csv(temporal_candidates_path)
selected_candidate, temporal_summary = reconcile_final_scope(
    scope_candidate_scorecard,
    temporal_candidate_comparison,
    temporal_selected_scope,
    selected_complaint_types=selected_complaint_types,
)
scope_candidate_scorecard["temporal_selected"] = scope_candidate_scorecard["candidate_id"].eq(
    selected_candidate["candidate_id"]
)
scope_candidate_scorecard.to_csv(
    TABLE_DIR / "scope_candidate_scorecard.csv", index=False
)
approved_candidates = scope_candidate_scorecard.loc[
    scope_candidate_scorecard["passes_critical_gates"]
]
approved_selection = bool(not approved_candidates.empty and selected_complaint_types)
display(scope_candidate_scorecard[[
    "candidate_id", "candidate_name", "eligible_target_count",
    "due_date_coverage", "closed_date_coverage", "missed_target_rate",
    "active_months", "missing_month_count", "invalid_timestamp_rate",
    "scope_score", "passes_critical_gates", "temporal_selected",
]].round(4))

,candidate_id,candidate_name,eligible_target_count,due_date_coverage,closed_date_coverage,missed_target_rate,active_months,missing_month_count,invalid_timestamp_rate,scope_score,passes_critical_gates,temporal_selected
0,C1,2022-01-01 through 2026-06-30,66340,0.9120,0.9862,0.5166,54,0,0.0,0.9105,True,False
1,C2,2022-01-01 through 2025-12-31,60281,0.9108,0.9944,0.5335,48,0,0.0,0.9055,True,False
2,C4,2023-01-01 through 2025-12-31,48427,0.9086,0.9946,0.4374,36,0,0.0,0.8933,True,False
3,C3,2023-01-01 through 2026-06-30,54486,0.9104,0.9847,0.4276,42,0,0.0,0.8904,True,False
4,C6,2024-01-01 through 2025-12-31,35966,0.9055,0.9932,0.4370,24,0,0.0,0.8881,True,True
5,C5,2024-01-01 through 2026-06-30,42025,0.9082,0.9806,0.4243,30,0,0.0,0.8846,True,False


**Interpretation.** The general scorecard evaluates data feasibility, coverage,
target balance, volume, continuity, timestamp consistency, and operational
suitability. Agency-wide percentages do not veto a viable subgroup.

The final date-range decision is delegated to Notebook 05 because it additionally
evaluates target-rate volatility, complete-calendar-year boundaries, outcome
maturity, and recent-period relevance. Therefore, the candidate with the highest
general `scope_score` is not automatically the final modelling period. The
`temporal_selected` column makes the authoritative reconciliation inspectable.

## 22. Temporal stability analysis

In [19]:
selected_start = pd.Timestamp(selected_candidate["start_date"])
selected_end = pd.Timestamp(selected_candidate["end_date"])
selected_monthly_raw = provisional_monthly.loc[provisional_monthly["created_month"].dt.tz_localize(None).between(selected_start, selected_end + pd.Timedelta(days=1), inclusive="left")]
monthly_selected_scope = selected_monthly_raw.groupby("created_month").agg({
    "total_records": "sum", "due_date_present_count": "sum", "closed_date_present_count": "sum",
    "eligible_target_count": "sum", "missed_target_count": "sum", "complaint_type": "nunique",
}).rename(columns={"complaint_type": "complaint_type_count"}).reset_index()
monthly_selected_scope["due_date_coverage"] = monthly_selected_scope["due_date_present_count"] / monthly_selected_scope["total_records"]
monthly_selected_scope["closed_date_coverage"] = monthly_selected_scope["closed_date_present_count"] / monthly_selected_scope["total_records"]
monthly_selected_scope["eligible_target_rate"] = monthly_selected_scope["eligible_target_count"] / monthly_selected_scope["total_records"]
monthly_selected_scope["on_time_count"] = monthly_selected_scope["eligible_target_count"] - monthly_selected_scope["missed_target_count"]
monthly_selected_scope["missed_target_rate"] = np.where(monthly_selected_scope["eligible_target_count"].gt(0), monthly_selected_scope["missed_target_count"] / monthly_selected_scope["eligible_target_count"], 0.0)
expected_months = pd.period_range(selected_start.to_period("M"), selected_end.to_period("M"), freq="M")
stability_indicators = summarize_monthly_stability(
    monthly_selected_scope,
    expected_month_count=len(expected_months),
)
stability_indicators["months_below_minimum_volume"] = int(
    monthly_selected_scope["eligible_target_count"].lt(MIN_COMPLAINT_TYPE_MEDIAN_MONTHLY_VOLUME).sum()
)
stability_indicators["months_below_required_coverage"] = int((
    monthly_selected_scope["due_date_coverage"].lt(MIN_AGENCY_DUE_DATE_COVERAGE)
    | monthly_selected_scope["closed_date_coverage"].lt(MIN_AGENCY_CLOSED_DATE_COVERAGE)
).sum())
# Written under a "general_" prefix: these are this notebook's own general
# evidence over the reconciled range, kept separate from the authoritative
# monthly/yearly tables Notebook 05 writes under the bare filenames.
monthly_selected_scope.to_csv(TABLE_DIR / "general_selected_scope_monthly_metrics.csv", index=False)
yearly_selected_scope = monthly_selected_scope.assign(year=monthly_selected_scope["created_month"].dt.year).groupby("year").agg(
    total_records=("total_records", "sum"), eligible_target_count=("eligible_target_count", "sum"),
    due_date_present_count=("due_date_present_count", "sum"), closed_date_present_count=("closed_date_present_count", "sum"),
    missed_target_count=("missed_target_count", "sum"),
).reset_index()
yearly_selected_scope["due_date_coverage"] = yearly_selected_scope["due_date_present_count"] / yearly_selected_scope["total_records"]
yearly_selected_scope["closed_date_coverage"] = yearly_selected_scope["closed_date_present_count"] / yearly_selected_scope["total_records"]
yearly_selected_scope["missed_target_rate"] = yearly_selected_scope["missed_target_count"] / yearly_selected_scope["eligible_target_count"]
yearly_selected_scope.to_csv(TABLE_DIR / "general_selected_scope_yearly_metrics.csv", index=False)
display(pd.DataFrame([stability_indicators]).T.rename(columns={0: "value"}).round(4))

,value
active_months,24
missing_month_count,0
monthly_volume_coefficient_of_variation,0.168325
minimum_monthly_eligible_volume,1219
maximum_monthly_eligible_volume,2367
target_rate_range,0.302285
largest_month_to_month_target_rate_change,0.215554
due_date_coverage_range,0.099118
closed_date_coverage_range,0.023194
target_rate_volatility_flag,True


### Temporal stability interpretation

These metrics describe the final population selected by the volatility-aware
temporal-stability analysis. Some wider periods receive higher general
feasibility scores because they contain more records. They are not selected
because the temporal analysis identifies materially greater target-rate
volatility, incomplete-period concerns, or a more stable contained candidate.
Missing months, low-volume months, and monthly coverage variation remain visible.
Continuity (zero missing months) is reported separately from stability: a
large `target_rate_range` or `largest_month_to_month_target_rate_change`
means the monthly missed-target rate is not homogeneous across the window,
and that must carry into the decision document and Month 2's validation
design rather than being inferred from continuity alone.

The monthly/yearly tables built in this section are exported as
`general_selected_scope_monthly_metrics.csv` and
`general_selected_scope_yearly_metrics.csv` — recomputed from this notebook's
own cached general evidence over the reconciled date range, for cross-checking.
The authoritative `selected_scope_monthly_metrics.csv` and
`selected_scope_yearly_metrics.csv` in this directory are written by
Notebook 05 from its fresh extraction and are the ones referenced by
`docs/scope_decision.md`.

## 23. Sensitivity analysis

In [20]:
baseline_sensitivity = rebuild_scoped_sensitivity_candidate(
    complaint_monthly,
    agency=provisional_agency,
    agency_name=provisional_agency_name,
    start_date=selected_start,
    end_date=selected_end,
    minimum_candidate_eligible_records=MIN_AGENCY_ELIGIBLE_RECORDS,
    minimum_complaint_type_eligible_records=MIN_COMPLAINT_TYPE_ELIGIBLE_RECORDS,
    minimum_due_date_coverage=MIN_AGENCY_DUE_DATE_COVERAGE,
    minimum_closed_date_coverage=MIN_AGENCY_CLOSED_DATE_COVERAGE,
    minimum_target_rate=MIN_TARGET_RATE,
    maximum_target_rate=MAX_TARGET_RATE,
    minimum_active_months=MIN_AGENCY_ACTIVE_MONTHS,
    minimum_median_monthly_volume=MIN_COMPLAINT_TYPE_MEDIAN_MONTHLY_VOLUME,
    maximum_invalid_timestamp_rate=MAX_INVALID_TIMESTAMP_RATE,
)
scenario_definitions = []
for value in (5_000, 10_000, 20_000):
    scenario_definitions.append((
        f"minimum_candidate_eligible_records_{value}",
        {"minimum_candidate_eligible_records": value},
    ))
for value in (0.60, 0.70, 0.80):
    scenario_definitions.append((
        f"minimum_due_date_coverage_{value:.0%}",
        {"minimum_due_date_coverage": value},
    ))
for value in (0.70, 0.80, 0.90):
    scenario_definitions.append((
        f"minimum_closed_date_coverage_{value:.0%}",
        {"minimum_closed_date_coverage": value},
    ))
for value in (500, 1_000, 2_500):
    scenario_definitions.append((
        f"minimum_complaint_type_eligible_records_{value}",
        {"minimum_complaint_type_eligible_records": value},
    ))
for label, year_shift in [
    ("selected_start_date", 0),
    ("one_year_earlier", -1),
    ("one_year_later", 1),
]:
    scenario_definitions.append((
        f"start_date_{label}",
        {"start_date": selected_start + pd.DateOffset(years=year_shift)},
    ))

sensitivity_rows = []
default_parameters = {
    "start_date": selected_start,
    "end_date": selected_end,
    "minimum_candidate_eligible_records": MIN_AGENCY_ELIGIBLE_RECORDS,
    "minimum_complaint_type_eligible_records": MIN_COMPLAINT_TYPE_ELIGIBLE_RECORDS,
    "minimum_due_date_coverage": MIN_AGENCY_DUE_DATE_COVERAGE,
    "minimum_closed_date_coverage": MIN_AGENCY_CLOSED_DATE_COVERAGE,
    "minimum_target_rate": MIN_TARGET_RATE,
    "maximum_target_rate": MAX_TARGET_RATE,
    "minimum_active_months": MIN_AGENCY_ACTIVE_MONTHS,
    "minimum_median_monthly_volume": MIN_COMPLAINT_TYPE_MEDIAN_MONTHLY_VOLUME,
    "maximum_invalid_timestamp_rate": MAX_INVALID_TIMESTAMP_RATE,
}
for scenario_name, overrides in scenario_definitions:
    parameters = {**default_parameters, **overrides}
    result = rebuild_scoped_sensitivity_candidate(
        complaint_monthly,
        agency=provisional_agency,
        agency_name=provisional_agency_name,
        **parameters,
    )
    sensitivity_rows.append({
        "scenario_name": scenario_name,
        "agency": result["agency"],
        "included_complaint_type_count": result["complaint_type_count"],
        "included_complaint_types": result["included_complaint_types"],
        "start_date": result["start_date"],
        "end_date": result["end_date"],
        "eligible_target_count": result["eligible_target_count"],
        "due_date_coverage": result["due_date_coverage"],
        "closed_date_coverage": result["closed_date_coverage"],
        "missed_target_rate": result["missed_target_rate"],
        "passes_critical_gates": result["passes_critical_gates"],
        "decision_status": determine_scope_decision_status(
            bool(result["passes_critical_gates"]),
            semantic_limitations_remain=True,
        ),
        "selection_changed": (
            result["included_complaint_types"]
            != baseline_sensitivity["included_complaint_types"]
            or result["start_date"] != baseline_sensitivity["start_date"]
            or bool(result["passes_critical_gates"])
            != bool(baseline_sensitivity["passes_critical_gates"])
        ),
    })
sensitivity_analysis = pd.DataFrame(sensitivity_rows)
sensitivity_analysis.to_csv(
    TABLE_DIR / "sensitivity_analysis.csv", index=False
)
display(sensitivity_analysis)

,scenario_name,agency,included_complaint_type_count,included_complaint_types,start_date,end_date,eligible_target_count,due_date_coverage,closed_date_coverage,missed_target_rate,passes_critical_gates,decision_status,selection_changed
0,minimum_candidate_eligible_records_5000,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False
1,minimum_candidate_eligible_records_10000,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False
2,minimum_candidate_eligible_records_20000,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False
3,minimum_due_date_coverage_60%,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False
4,minimum_due_date_coverage_70%,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False
5,minimum_due_date_coverage_80%,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False
6,minimum_closed_date_coverage_70%,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False
7,minimum_closed_date_coverage_80%,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False
8,minimum_closed_date_coverage_90%,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False
9,minimum_complaint_type_eligible_records_500,DSNY,1,Graffiti,2024-01-01,2025-12-31,35966,0.905515,0.993228,0.436968,True,Approved with limitations,False


### Sensitivity analysis interpretation

Every scenario reselects complaint types and rebuilds the filtered candidate.
The table records when coverage, volume, date boundaries, included types, or
approval status change; it never reuses an agency-wide feasibility boolean.

## 24. Final scope recommendation

In [21]:
decision_status = str(temporal_summary["decision_status"])
agency_wide_due_date_coverage = float(
    agency_scope_comparison.loc[
        agency_scope_comparison["agency"] == provisional_agency,
        "due_date_coverage",
    ].iloc[0]
)
selected_scope = {
    "selected_agency": str(temporal_summary["selected_agency"]),
    "selected_agency_name": provisional_agency_name,
    "selected_start_date": str(temporal_summary["selected_start_date"]),
    "selected_end_date": str(temporal_summary["selected_end_date"]),
    "selected_complaint_type_count": len(selected_complaint_types),
    "selected_complaint_type": str(temporal_summary["selected_complaint_type"]),
    "selected_complaint_types": " | ".join(selected_complaint_types),
    "total_records": int(selected_candidate["total_records"]),
    "eligible_target_records": int(selected_candidate["eligible_target_count"]),
    "eligibility_rate": float(selected_candidate["eligible_target_rate"]),
    "missed_target_count": int(selected_candidate["missed_target_count"]),
    "on_time_count": int(selected_candidate["on_time_count"]),
    "missed_target_rate": float(selected_candidate["missed_target_rate"]),
    "due_date_coverage": float(selected_candidate["due_date_coverage"]),
    "closed_date_coverage": float(selected_candidate["closed_date_coverage"]),
    "agency_wide_due_date_coverage": agency_wide_due_date_coverage,
    "active_months": int(selected_candidate["active_months"]),
    "missing_month_count": int(selected_candidate["missing_month_count"]),
    "scope_score": float(selected_candidate["scope_score"]),
    "decision_status": decision_status,
    "complaint_type_evidence_start_date": ANALYSIS_START_DATE,
    "complaint_type_evidence_end_date": latest_complete_month_end.date().isoformat(),
    **{f"stability_{key}": value for key, value in stability_indicators.items()},
}
validate_selected_scope_consistency(selected_scope, selected_complaint_types)
display(pd.DataFrame([selected_scope]).T.rename(columns={0: "value"}))

,value
selected_agency,DSNY
selected_agency_name,Department of Sanitation
selected_start_date,2024-01-01
selected_end_date,2025-12-31
selected_complaint_type_count,1
...,...
stability_due_date_coverage_range,0.099118
stability_closed_date_coverage_range,0.023194
stability_target_rate_volatility_flag,True
stability_months_below_minimum_volume,0


### Final decision rationale

The DSNY Graffiti subgroup passes complaint-level feasibility gates. Date-range
approval is narrowed to 2024-01-01 through 2025-12-31 by Notebook 05 because
the wider candidates have severe temporal volatility. The status remains
**Approved with limitations** because moderate temporal variation and target
governance limitations remain.

## 25. Rejected alternatives

In [22]:
rejected_scope_candidates = scope_candidate_scorecard.loc[
    scope_candidate_scorecard["candidate_id"] != selected_candidate["candidate_id"]
].copy()
rejected_scope_candidates = rejected_scope_candidates.merge(
    temporal_candidate_comparison[[
        "start_date", "end_date", "decision_status", "decision_reasons",
        "temporal_volatility_warning", "severe_temporal_volatility",
        "outcome_immature_count", "selected",
    ]],
    on=["start_date", "end_date"],
    how="left",
    validate="one_to_one",
)
if rejected_scope_candidates["decision_status"].isna().any():
    raise ValueError("Every rejected general candidate requires temporal evidence.")
rejected_scope_candidates["date_range"] = (
    rejected_scope_candidates["start_date"].astype(str)
    + " to "
    + rejected_scope_candidates["end_date"].astype(str)
)
rejected_scope_candidates["decision"] = "Rejected by volatility-aware temporal analysis"

def temporal_rejection_reasons(row: pd.Series) -> str:
    """Explain rejection using only recorded temporal candidate evidence."""
    reasons = []
    if pd.notna(row["decision_reasons"]) and str(row["decision_reasons"]).strip():
        reasons.append(str(row["decision_reasons"]))
    end_date = pd.Timestamp(row["end_date"])
    if (end_date.month, end_date.day) != (12, 31):
        reasons.append("Includes an incomplete calendar year")
    if int(row["outcome_immature_count"]) > 0:
        reasons.append("Contains outcome-immature records")
    if bool(row["temporal_volatility_warning"]) and not bool(row["severe_temporal_volatility"]):
        reasons.append("Passes core feasibility gates but is materially less stable")
    if not reasons:
        reasons.append("Rejected by volatility-aware temporal analysis")
    return " | ".join(dict.fromkeys(reasons))

rejected_scope_candidates["rejection_reasons"] = rejected_scope_candidates.apply(
    temporal_rejection_reasons, axis=1
)
rejected_scope_candidates = rejected_scope_candidates[[
    "candidate_id", "candidate_name", "agency", "date_range",
    "complaint_type_count", "eligible_target_count", "due_date_coverage",
    "closed_date_coverage", "missed_target_rate", "decision",
    "decision_status", "rejection_reasons", "scope_score",
]]
# Written under a "general_" prefix: Notebook 05 independently writes the
# authoritative rejected_scope_candidates.csv into this same directory.
rejected_scope_candidates.to_csv(
    TABLE_DIR / "general_rejected_scope_candidates.csv", index=False
)
display(rejected_scope_candidates.round(4))

,candidate_id,candidate_name,agency,date_range,complaint_type_count,eligible_target_count,due_date_coverage,closed_date_coverage,missed_target_rate,decision,decision_status,rejection_reasons,scope_score
0,C1,2022-01-01 through 2026-06-30,DSNY,2022-01-01 to 2026-06-30,1,66340,0.9120,0.9862,0.5166,Rejected by volatility-aware temporal analysis,REQUIRES_NARROWER_RANGE,"Severe temporal volatility and a feasible, mor...",0.9105
1,C2,2022-01-01 through 2025-12-31,DSNY,2022-01-01 to 2025-12-31,1,60281,0.9108,0.9944,0.5335,Rejected by volatility-aware temporal analysis,REQUIRES_NARROWER_RANGE,"Severe temporal volatility and a feasible, mor...",0.9055
2,C4,2023-01-01 through 2025-12-31,DSNY,2023-01-01 to 2025-12-31,1,48427,0.9086,0.9946,0.4374,Rejected by volatility-aware temporal analysis,APPROVED_WITH_LIMITATIONS,Passes core feasibility gates but is materiall...,0.8933
3,C3,2023-01-01 through 2026-06-30,DSNY,2023-01-01 to 2026-06-30,1,54486,0.9104,0.9847,0.4276,Rejected by volatility-aware temporal analysis,REQUIRES_NARROWER_RANGE,"Severe temporal volatility and a feasible, mor...",0.8904
4,C5,2024-01-01 through 2026-06-30,DSNY,2024-01-01 to 2026-06-30,1,42025,0.9082,0.9806,0.4243,Rejected by volatility-aware temporal analysis,APPROVED_WITH_LIMITATIONS,Includes an incomplete calendar year | Contain...,0.8846


### Rejected alternatives rationale

Rejection reasons come from Notebook 05's temporal candidate evidence, not the
general feasibility ranking. Wider candidates may have higher `scope_score`
because they contain more data while still being rejected for severe volatility,
incomplete calendar years, outcome immaturity, or materially weaker stability.

This table is exported as `general_rejected_scope_candidates.csv` to avoid
overwriting Notebook 05's authoritative `rejected_scope_candidates.csv` in
the same directory.

## 26. Exported outputs

Filenames shared with Notebook 05 (`selected_scope_summary.csv`,
`date_range_candidate_comparison.csv`, `selected_scope_monthly_metrics.csv`,
`selected_scope_yearly_metrics.csv`, `rejected_scope_candidates.csv`,
`complaint_type_inclusion_evidence.csv`) are authoritative only when written
by Notebook 05. This notebook's own general-evidence versions of the latter
five are exported separately under a `general_` prefix so the two independent
computations never silently overwrite each other under the same filename.

In [23]:
selected_scope_summary = temporal_selected_scope.copy()
selected_scope_summary.to_csv(
    TABLE_DIR / "selected_scope_summary.csv", index=False
)
validate_selected_scope_outputs(
    pd.read_csv(TABLE_DIR / "selected_scope_summary.csv"),
    temporal_selected_scope,
)

def save_agency_chart(column: str, filename: str, title: str, percentage: bool = False, threshold: float | None = None) -> None:
    chart = agency_scope_comparison.nlargest(15, column).sort_values(column)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(chart["agency"].astype(str), chart[column], color="#4472C4")
    ax.set_title(title)
    if percentage:
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
        ax.set_xlim(0, 1)
    if threshold is not None:
        ax.axvline(threshold, color="#C00000", linestyle="--", label=f"Diagnostic: {threshold:.0%}")
        ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)

save_agency_chart("eligible_target_count", "eligible_records_by_agency.png", "Target-eligible records by agency")
save_agency_chart("due_date_coverage", "due_date_coverage_by_agency.png", "Agency-wide due-date coverage (diagnostic)", True, MIN_AGENCY_DUE_DATE_COVERAGE)
save_agency_chart("closed_date_coverage", "closed_date_coverage_by_agency.png", "Agency-wide closed-date coverage (diagnostic)", True, MIN_AGENCY_CLOSED_DATE_COVERAGE)
save_agency_chart("missed_target_rate", "missed_target_rate_by_agency.png", "Missed-target rate by agency", True)

def save_monthly_line(column: str, filename: str, title: str, percentage: bool = False) -> None:
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(monthly_selected_scope["created_month"], monthly_selected_scope[column], linewidth=1.8)
    ax.set_title(title + " (selected scoped population)")
    if percentage:
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.set_ylim(0, 1)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)

save_monthly_line("eligible_target_count", "monthly_volume_selected_scope.png", "Monthly eligible volume")
save_monthly_line("missed_target_rate", "monthly_target_rate_selected_scope.png", "Monthly missed-target rate", True)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly_selected_scope["created_month"], monthly_selected_scope["due_date_coverage"], label="Due-date coverage")
ax.plot(monthly_selected_scope["created_month"], monthly_selected_scope["closed_date_coverage"], label="Closed-date coverage")
ax.axhline(MIN_AGENCY_DUE_DATE_COVERAGE, color="#C00000", linestyle="--", label="Candidate due-date gate")
ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_ylim(0, 1); ax.legend(); ax.set_title("Monthly coverage (selected scoped population)")
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(FIGURE_DIR / "monthly_coverage_selected_scope.png", dpi=150, bbox_inches="tight"); plt.close(fig)

required_tables = [
    "agency_scope_comparison.csv", "complaint_type_scope_comparison.csv",
    "date_range_candidate_comparison.csv", "scope_candidate_scorecard.csv",
    "selected_scope_summary.csv", "complaint_type_inclusion_evidence.csv",
    "rejected_scope_candidates.csv", "sensitivity_analysis.csv",
    "selected_scope_monthly_metrics.csv", "selected_scope_yearly_metrics.csv",
    "general_complaint_type_inclusion_evidence.csv",
    "general_date_range_candidate_comparison.csv",
    "general_rejected_scope_candidates.csv",
    "general_selected_scope_monthly_metrics.csv",
    "general_selected_scope_yearly_metrics.csv",
]
required_figures = [
    "eligible_records_by_agency.png", "due_date_coverage_by_agency.png",
    "closed_date_coverage_by_agency.png", "missed_target_rate_by_agency.png",
    "monthly_volume_selected_scope.png", "monthly_target_rate_selected_scope.png",
    "monthly_coverage_selected_scope.png",
]
display(pd.DataFrame({"output": [
    *(str((TABLE_DIR / name).relative_to(PROJECT_ROOT)) for name in required_tables),
    *(str((FIGURE_DIR / name).relative_to(PROJECT_ROOT)) for name in required_figures),
]}))

,output
0,reports/04_scope_selection/tables/agency_scope...
1,reports/04_scope_selection/tables/complaint_ty...
2,reports/04_scope_selection/tables/date_range_c...
3,reports/04_scope_selection/tables/scope_candid...
4,reports/04_scope_selection/tables/selected_sco...
5,reports/04_scope_selection/tables/complaint_ty...
6,reports/04_scope_selection/tables/rejected_sco...
7,reports/04_scope_selection/tables/sensitivity_...
8,reports/04_scope_selection/tables/selected_sco...
9,reports/04_scope_selection/tables/selected_sco...


## 27. Scope decision document

In [24]:
extraction_values = {
    "dataset_id": DATASET_ID, "source": API_ENDPOINT,
    "extraction_timestamp": extraction_timestamp.isoformat(),
    "requested_start_date": ANALYSIS_START_DATE,
    "requested_end_date": ANALYSIS_END_DATE or "live snapshot",
    "rows_represented": rows_represented,
    "latest_available_date": latest_available_date.isoformat(),
    "latest_complete_month": latest_complete_month_end.strftime("%Y-%m"),
}
limitations = [
    "The initial scope contains one complaint type, so complaint_type is constant and cannot be a varying model feature.",
    "The business meaning and creation-time semantics of due_date require confirmation.",
    "Prediction-time availability and post-creation changes to due_date require confirmation.",
    "Status eligibility and cancelled/duplicate complaint treatment remain to be approved.",
    "A scoped row-level extraction must validate identifiers, parsing, chronology, target construction, and status distributions before modelling.",
    "Operational relevance has no objective repository-backed measure and is scored neutrally.",
]
display(Markdown(
    "Notebook 05 owns the final outcome-maturity and temporal date-range decision. "
    "See `docs/scope_decision.md` after executing that notebook."
))

Notebook 05 owns the final outcome-maturity and temporal date-range decision. See `docs/scope_decision.md` after executing that notebook.

## 28. Completion assertions

In [25]:
assert not agency_scope_comparison.empty
assert not complaint_type_scope_comparison.empty
assert not date_range_comparison.empty
assert not scope_candidate_scorecard.empty
assert selected_scope["selected_complaint_type_count"] == len(selected_complaint_types)
if approved_selection:
    assert selected_scope["eligible_target_records"] > 0
    assert selected_scope["selected_complaint_type_count"] > 0
    assert selected_scope["selected_start_date"] < selected_scope["selected_end_date"]
    assert selected_scope["due_date_coverage"] >= MIN_AGENCY_DUE_DATE_COVERAGE
    assert selected_scope["closed_date_coverage"] >= MIN_AGENCY_CLOSED_DATE_COVERAGE
    assert MIN_TARGET_RATE <= selected_scope["missed_target_rate"] <= MAX_TARGET_RATE
    assert selected_scope["missing_month_count"] == 0
    assert selected_scope["decision_status"] in {
        "APPROVED", "APPROVED_WITH_LIMITATIONS"
    }
else:
    assert selected_scope["decision_status"] == "Requires review"
assert all((TABLE_DIR / name).is_file() for name in required_tables)
assert all((FIGURE_DIR / name).is_file() for name in required_figures)
assert selected_scope["selected_start_date"] == str(temporal_summary["selected_start_date"])
assert selected_scope["selected_end_date"] == str(temporal_summary["selected_end_date"])
assert selected_scope["selected_agency"] == str(temporal_summary["selected_agency"])
assert selected_scope["selected_complaint_type"] == str(temporal_summary["selected_complaint_type"])
assert int(scope_candidate_scorecard["temporal_selected"].sum()) == 1

## 29. Final notebook conclusion

The selected modelling population is not all NYC 311 complaints and not all
DSNY complaints. It is limited to the complaint types selected by the
reproducible Stage 2 rules—currently DSNY Graffiti—during the selected date
range and satisfying Notebook 03 target eligibility.

The scoped population is **Approved with limitations** for 2024-01-01 through
2025-12-31. Continuity and stability are separate: all 24 expected months are
active, while target-rate variation remains a downstream temporal risk. Due-date
meaning and prediction-time availability, status eligibility, cancelled and
duplicate treatment, and row-level quality must be resolved before modelling.
Because one complaint type is selected, `complaint_type` is constant and is
not useful as a varying feature in the initial model.

**Next data-readiness gate.** Retrieve the selected agency, complaint type,
and date range at row level and confirm unique-key uniqueness, conflicting
duplicates, timestamp parsing, `created_date <= due_date`, `created_date <=
closed_date`, target construction, status distribution, cancellation and
duplicate rules, creation-time availability of `due_date`, and whether that
field changes after complaint creation. This notebook does not build the
modelling dataset or train a model.